# KinoCompass – EASE на GPU

EASE (Steck, 2019) – линейная модель «фильм × фильм» с решением в одну формулу. На всех ~21 тыс. фильмах-кандидатах её матрица весит около 2 ГБ, поэтому она обучается здесь, на GPU.

Ноутбук клонирует репозиторий и запускает **тот же** код проверки, что и в Docker (`model/evaluate.py`, `protocol.py`, `metrics.py`): те же люди, те же сценарии, те же метрики. Набор MovieLens 32M скачивается в память Colab.

**Как запустить:** «Среда выполнения» → «Сменить среду выполнения» → **T4 GPU**, затем «Среда выполнения» → «Выполнить все». В конце скачается архив `ease_<период>.zip` с отчётом (около 100 КБ) – его нужно положить в `model/reports/`.

Время на T4 (бесплатный Colab): загрузка набора и сценарии – около 2 минут, пять значений λ – около 2 минут на обучение и нескольких минут на проверку каждое, всего около 20–30 минут. Вкладку не закрывать до конца.

In [ ]:
# проверочная часть: validation – выбор настроек; test – только на шаге 6, один раз
PERIOD = "validation"

In [ ]:
import torch

assert torch.cuda.is_available(), "нет GPU: Среда выполнения -> Сменить среду выполнения -> T4 GPU"
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "ГБ")

In [ ]:
!rm -rf /content/KinoCompass && git clone -q --depth 1 https://github.com/AlexTrav/KinoCompass.git /content/KinoCompass
!git -C /content/KinoCompass log --oneline -1
# те же версии, что в образе Docker (torch в Colab уже есть)
!pip install -q requests==2.34.2 pandas==3.1.0 pyarrow==26.0.1 scipy==1.18.1

In [ ]:
%cd /content/KinoCompass/model
!PYTHONPATH=/content/KinoCompass/data:/content/KinoCompass/research PERIOD={PERIOD} python -W ignore evaluate.py ease

In [ ]:
from google.colab import files

!cd /content/KinoCompass/model && zip -q -r /content/ease_{PERIOD}.zip reports/ease && ls -la reports/ease
files.download(f"/content/ease_{PERIOD}.zip")